In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 

In [ ]:
ls /home/luiz/desktop/UFPA/2026.2/lab_eng_software/Projeto_vale/data/telemetry/processed

In [2]:
seed = 42
path = "/home/luiz/desktop/UFPA/2026.2/lab_eng_software/Projeto_vale/data/telemetry/processed"
january = os.path.join(path, "jan.csv")
jan = pd.read_csv(january)

path_alarmes = "/home/luiz/desktop/UFPA/2026.2/lab_eng_software/Projeto_vale/Material/"
alarmes = os.path.join(path_alarmes, "Alarmes-Regra-de-Negocio.csv")
alar = pd.read_csv(alarmes)

/tmp/ipykernel_32545/1494084696.py:4: DtypeWarning: Columns (0: Valor, 1: Classe) have mixed types. Specify dtype option on import or set low_memory=False.
  jan = pd.read_csv(january)


In [ ]:
head_jan = jan.head()
head_alarmes = alar.head()
#head_jan
head_alarmes


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

# 1. Lendo os dados
df = jan
# Pegando as primeiras linhas
df_to_plot = df.head(10)

# --- TRANSPOSIÇÃO DOS DADOS ---
# Transpõe o DataFrame (.T) e transforma o índice (nomes das colunas originais) em uma coluna real
df_transposto = df_to_plot.T.reset_index()

# Configuração do tamanho da figura (ajustado para o formato vertical da transposição)
fig, ax = plt.subplots(figsize=(120, 18))
ax.axis("tight")
ax.axis("off")

# 2. Criar a tabela centralizada
# Como transcrevemos tudo para o corpo da tabela, não passamos 'colLabels' para não duplicar o cabeçalho no topo
the_table = ax.table(
    cellText=df_transposto.values,
    loc="center",
    cellLoc="center",
)

# 3. Travar o tamanho da fonte e dar espaçamento vertical (scale)
the_table.auto_set_font_size(False)
the_table.set_fontsize(10)
the_table.scale(1.0, 2.0)  # Ajustado para 2.0 para manter um bom respiro vertical

# 4. Estilização Transposta (Coluna da esquerda escura, linhas alternadas no resto)
for (row, col), cell in the_table.get_celld().items():
    if col == 0:
        # Primeira coluna (antigo cabeçalho) fica em Azul Escuro
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#1F4E78")
    else:
        # Linhas alternadas em cinza claro para o restante dos dados
        if row % 2 == 0:
            cell.set_facecolor("#F2F2F2")
        else:
            cell.set_facecolor("white")

# 5. Salva no PDF primeiro
with PdfPages("alarmes_transposto.pdf") as pp:
    pp.savefig(fig, bbox_inches="tight", dpi=200)

# 6. MANDA PLOTAR NA TELA SUCESSO!
plt.show()

In [ ]:


def show_missig_values(dataset):
    rows = dataset.shape[0]
    missing_counts = dataset.isnull().sum().reset_index()
    missing_counts.columns = ["coluna", "valores_faltantes"]
    
    print("Número de valores faltando.")
    for col, val in dataset.isnull().sum().items():
        print(f"{col}: {val} -> {val/rows*100:.2f}%")
    
    plt.figure(figsize=(12,6))
    sns.barplot(data=missing_counts, 
                x="coluna", 
                y="valores_faltantes",
                hue="coluna",
                palette="viridis",
                legend=False,
                dodge=False)
    
    plt.xticks(rotation=45, ha='right')
    plt.title("Valores Faltando por Coluna", fontsize=16)
    plt.xlabel("Colunas")
    plt.ylabel("Número de valores faltantando")
    plt.tight_layout()
    plt.show()
#for i in months:
#    show_missig_values(i)
show_missig_values(jan)

In [5]:
jan = jan.drop(columns="Classe")
print(jan.shape)

(5400002, 13)


In [ ]:
def show_distribution(dataset, cut_columns=[], max_categories=50000):
    """
    Exibe a distribuição de colunas relevantes, filtrando IDs, 
    colunas constantes e alta cardinalidade.
    """
    for col in dataset.columns:
        # 1. Filtro de colunas ignoradas manualmente
        if col in cut_columns:
            continue
            
        # 2. Filtro de Variância: Ignora se a coluna tiver apenas 1 valor único (ex: Localidade "Itabira" em tudo)
        unique_count = dataset[col].nunique()
        if unique_count <= 1:
            continue
            
        # 3. Filtro de IDs/Alta Cardinalidade: Se for ID ou Hash (muitos valores únicos em relação ao total)
        # Se mais de 90% dos dados forem únicos, provavelmente é um ID/Chave Primária
        if unique_count > len(dataset) * 0.9:
            continue

        plt.figure(figsize=(10, 5))
        
        # Lógica para colunas NUMÉRICAS
        if pd.api.types.is_numeric_dtype(dataset[col]):
            plt.title(f"Distribuição Numérica: '{col}'", fontsize=12)
            sns.histplot(dataset[col], color="skyblue", bins=30, stat="density", kde=True)
            plt.ylabel("Densidade")
            
        # Lógica para colunas CATEGÓRICAS
        else:
            # Se tiver categorias demais (ex: nomes de operadores), pegamos apenas as Top X
            if unique_count > max_categories:
                data_to_plot = dataset[col].value_counts().head(max_categories)
                plt.title(f"Top {max_categories} Categorias: '{col}'", fontsize=12)
            else:
                data_to_plot = dataset[col].value_counts()
                plt.title(f"Distribuição Categórica: '{col}'", fontsize=12)

            sns.barplot(x=data_to_plot.index, y=data_to_plot.values, palette="viridis", hue=data_to_plot.index, legend=False)
            plt.xticks(rotation=45, ha="right")
            plt.ylabel("Frequência")

        plt.xlabel(col)
        plt.tight_layout()
        plt.show()

In [ ]:

def particoes(month: pandas.series, n_chunks: int) -> list:
    chunk_size = len(month) // n_chunks
    
    
    return [month.iloc[i * chunk_size : (i + 1) * chunk_size] for i in range(n_chunks)]
    



In [ ]:
partes = particoes(jan, 180)

print(partes[0])


In [ ]:
show_distribution(partes[0])

- Existem alarmes sem um valor correspondente, essas linhas podem ser descartadas?
- Features como inicio do turno e fim do turno podem ser agrupados em categorias como manha1, manha2, tarde1, tarde2?
- Classe tem 97% dos dados faltando, essa feature deve ser descartada?
- Ids devem ser considerados categóricos?
- O que eu devo fazer com alarme considerando que existem diversos tipos e é custoso visualizar todas as flags a olho nu?
- Quais features eu posso remover sem prejuízo?
- Posso agrupar diferentes tipos de "Alarme"? Se freio tem as possibilidades temperatura, pressão e auxiliar eu posso juntar tudo em freio ou devo separar em freio 1 para mais criticos e freios 2 para menos críticos?
- 

In [ ]:
a = jan["Alarme"].unique()
b = jan["Criticidade"].unique()
a2 = jan["Alarme"]
b2 = jan["Criticidade"]
c = jan["Is_Dont_Go"]
d = jan["Tipo"].unique()
a, b, a2, b2, c, d

In [ ]:
total = 0
casos = 0
for i in zip(b2, c):
    if i[0] == "Critico":
        total = total + 1
    if i[0] == "Critico" and i[1] == 1:
        casos = casos + 1
casos/total

7.7% dos casos críticos gera um dont go?
O que isso quer dizer, será que essa feature está correta ou eu estou interpretando errado?

In [3]:
jan= jan.drop(columns=['Id_Eventos_Telemetria'])
jan = jan.drop(columns=['Nome_Operador_Anon'])
jan = jan.drop(columns=['Dia'])


Removendo colunas consideradas redundantes. 
Id telemetria: identificador de linha, não causa impacto no modelo 
Nome do operador: como o hash é fornecido há uma redundância
Dia: existe data_evento, redundância

In [ ]:
print(jan.shape)
jan.head(5)

In [ ]:
contador = 0
for i in jan["Localidade"]:
    if i == "Itabira":
        contador = contador + 1
contador

Como todas medidass são em Itabira imagino que essa coluna não faz sentido ser mantida dado que, imagino, a correlação com o target seria sem 0

In [4]:
jan = jan.drop(columns=['Localidade'])


In [ ]:
tipos = jan["Tipo"].unique()
tipos

Caminhão e escavadeira são os dois únicos veículos que a telemetria acusa.
Devo checar se em outros meses há outro tipo de veículo, se não houver devo mudar a coluna para um boleano is_truck?


In [ ]:
print(jan.shape)

In [ ]:
def classificar_alarme_detalhado(nome_alarme):
    alarme = str(nome_alarme).upper()
    
    # ---------------------------------------------------------
    # 1. IDENTIFICAÇÃO DA SEVERIDADE (1 ou 2)
    # ---------------------------------------------------------
    # Palavras que indicam falha crítica, impacto direto ou parada
    criticos = ['MC -', 'MA -', 'ACTIVE', 'QUEDA', 'BAIXA PRESSÃO', 'FALHA', 'LOW PRESSURE', '>']
    
    # Palavras que indicam aviso, timeout, inatividade ou operação
    avisos = ['OP -', 'INACTIVE', 'TIMEOUT', 'NORMAL', 'CHANGE RATE', 'ABUSE', 'DUMP', 'LEVEL']
    
    # Define a severidade inicial
    severidade = "1" # Padrão é 1 (Aviso/Menos grave)
    if any(c in alarme for c in criticos) and not any(a in alarme for a in ['INACTIVE', 'NORMAL']):
        severidade = "2" # Se for crítico e NÃO estiver inativo/normal, vira 2

    # ---------------------------------------------------------
    # 2. IDENTIFICAÇÃO DO SISTEMA
    # ---------------------------------------------------------
    sistema = "Outros" # Categoria padrão se não achar nenhuma
    
    # Dicionário de sistemas e suas palavras-chave
    regras_sistemas = {
        "Motor": ['ENGINE', 'COOLANT', 'OIL', 'AFTERCOOLER', 'TURBO', 'EXHAUST', 'ARREF', 'COMBUSTÍVEL', 'PRELUBE', 'FAN'],
        "Freio": ['BRAKE', 'FREIO', 'IBC', 'ARC', 'RETARDO', 'PARKING'],
        "Transmissao": ['TRANSMISSION', 'TORQUE CONVERTER', 'SHIFT', 'DIFFERENTIAL', 'FINAL DRIVE', 'MARCHA'],
        "Suspensao_Lataria": ['SUSPENSION', 'STEERING', 'DIREÇÃO', 'CHASSIS'],
        "Pneu": ['TIRE', 'PNEU'],
        "Carga": ['PAYLOAD', 'DIPPER', 'LOAD', 'CYCLE', 'BODY UP'],
        "Eletronica": ['ECM', 'DATA LINK', 'VIMS', 'CHANNEL', 'OEM', 'VOLTAGEM', 'VOLTAGE', 'SENSOR']
    }
    
    # Busca qual sistema o alarme pertence
    for chave_sistema, palavras_chave in regras_sistemas.items():
        if any(palavra in alarme for palavra in palavras_chave):
            sistema = chave_sistema
            break # Achou o sistema, para de procurar
            
    # ---------------------------------------------------------
    # 3. RETORNA A COMBINAÇÃO (Ex: Freio2, Motor1, Pneu1)
    # ---------------------------------------------------------
    return f"{sistema}{severidade}"

In [ ]:
novo = []
for alarme in partes[0]["Alarme"]:
    novo.append(classificar_alarme_detalhado(alarme))
novo
#classificar_alarme(partes[0]["Alarme"])

Posso agrupar avisos operacionais e não criticos?

In [ ]:
head_jan = jan.head()
head_jan

# Exibe o head com cabeçalho destacado, linhas de grade explícitas e alinhamento centralizado
head_jan.style.set_properties(**{
    'background-color': 'white',
    'color': 'black',
    'border': '1px solid #D3D3D3',       # Adiciona linhas entre linhas e colunas
    'text-align': 'center',              # Centraliza o texto das células
    'padding': '8px'
}).set_table_styles([
    {
        'selector': 'th', 
        'props': [
            ('background-color', '#1F4E78'), 
            ('color', 'white'), 
            ('font-weight', 'bold'),
            ('border', '1px solid #D3D3D3'), # Adiciona as linhas no cabeçalho também
            ('text-align', 'center'),
            ('padding', '10px')
        ]
    }
])

In [ ]:
import matplotlib.pyplot as plt

# 1. Configurando a estrutura gráfica da tabela
fig, ax = plt.subplots(figsize=(50, 2.5)) # Ajuste o tamanho baseado na quantidade de colunas
ax.axis('tight')
ax.axis('off')

# 2. Criando a tabela integrada com os dados do dataframe
table = ax.table(
    cellText=head_jan.values,
    colLabels=head_jan.columns,
    cellLoc='center',
    loc='center'
)

# 3. Ajustando proporções de espaçamento e fontes
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.1, 1.8) # Alarga as linhas verticalmente para um aspecto profissional

# 4. Aplicando a estilização das bordas, cabeçalho e preenchimento
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#B0B0B0')  # Linhas divisórias cinzas explícitas
    cell.set_linewidth(1.2)        # Espessura da linha separadora
    
    if row == 0:
        # Cabeçalho Azul Escuro padrão do projeto
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#1F4E78')
    else:
        # Linhas de dados com sutil alternância de fundo para melhor leitura
        if row % 2 == 0:
            cell.set_facecolor('#F9FBFD')
        else:
            cell.set_facecolor('#FFFFFF')

# 5. Salvando nos formatos solicitados com alta resolução
plt.savefig('head_janeiro.png', bbox_inches='tight', dpi=300)
plt.savefig('head_janeiro.svg', bbox_inches='tight')
plt.close()

print("Imagens 'head_janeiro.png' e 'head_janeiro.svg' geradas com sucesso!")

In [6]:
# Substitua o 'caminho_e_nome_do_arquivo.csv' pelo local onde deseja salvar
jan.to_csv('jan_limpo.csv', index=False)